# Iris Flower Classification with K-Nearest Neighbors (KNN)

This notebook walks through a complete supervised machine learning pipeline on the classic **Iris** dataset using the **K-Nearest Neighbors** algorithm.

**Pipeline stages:** load → explore → scale → split → build → train → predict → evaluate → visualize → tune `k`.

Every stage is explained so it is easy to follow even if you are new to machine learning.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
)

# Make plots appear inline in the notebook.
%matplotlib inline
RANDOM_STATE = 42  # a fixed seed so results are reproducible

## 1. Load the dataset

We load Iris directly from scikit-learn and inspect its feature names, target classes, and shape.

In [ ]:
iris = load_iris()

print('Feature names:', list(iris.feature_names))
print('Target classes:', list(iris.target_names))
print('Data shape (rows, columns):', iris.data.shape)

In [ ]:
# Convert to a pandas DataFrame for easy exploration.
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = [iris.target_names[i] for i in iris.target]
df.head()

## 2. Explore the data

Preview the statistics, confirm there are no missing values, and check the class balance.

In [ ]:
df.describe()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nSamples per species:')
print(df['species'].value_counts())

### Feature scatter plots

These plots show how the three species separate. Petal measurements separate the classes especially cleanly, which is why KNN does so well here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colours = ['#1f77b4', '#ff7f0e', '#2ca02c']
for colour, sp in zip(colours, df['species'].unique()):
    sub = df[df['species'] == sp]
    axes[0].scatter(sub[iris.feature_names[0]], sub[iris.feature_names[1]], label=sp, color=colour, alpha=0.7)
    axes[1].scatter(sub[iris.feature_names[2]], sub[iris.feature_names[3]], label=sp, color=colour, alpha=0.7)
axes[0].set(xlabel=iris.feature_names[0], ylabel=iris.feature_names[1], title='Sepal length vs width')
axes[1].set(xlabel=iris.feature_names[2], ylabel=iris.feature_names[3], title='Petal length vs width')
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Feature scaling

KNN measures distances between points, so features must be on the same scale. `StandardScaler` gives every feature a mean of 0 and a standard deviation of 1. We scale only the input features `X`, never the labels `y`.

In [ ]:
X = df[iris.feature_names]
y = df['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Mean after scaling (~0):', np.round(X_scaled.mean(axis=0), 2))
print('Std after scaling  (~1):', np.round(X_scaled.std(axis=0), 2))

## 4. Train / test split

80% of the data trains the model; 20% tests it on flowers it has never seen. `stratify=y` keeps the class balance in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, shuffle=True, random_state=RANDOM_STATE, stratify=y
)
print('Training samples:', X_train.shape[0])
print('Testing samples: ', X_test.shape[0])

## 5-7. Build, train, and predict (k = 5)

We start with `k = 5`: to classify a flower, KNN looks at its 5 nearest neighbours and takes a majority vote.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)          # 'training' = memorising the training points
y_pred = knn.predict(X_test)       # predict the unseen test flowers
y_pred[:10]

## 8. Evaluate the model

Accuracy, a confusion matrix, and per-class precision / recall / F1.

In [ ]:
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred, average='macro'), 4))
print('Recall   :', round(recall_score(y_test, y_pred, average='macro'), 4))
print('F1 score :', round(f1_score(y_test, y_pred, average='macro'), 4))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### 9. Confusion matrix plot

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
fig.colorbar(im, ax=ax, label='Number of samples')
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(iris.target_names, rotation=45, ha='right')
ax.set_yticklabels(iris.target_names)
ax.set(xlabel='Predicted label', ylabel='True label', title='Confusion Matrix (k=5)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

## 10. K value experiment

Now we test every `k` from 1 to 20 and plot the accuracy for each. Small `k` can overfit (jagged boundary); larger `k` gives a smoother boundary. When several `k` tie on accuracy we prefer the largest for robustness.

In [ ]:
k_values = list(range(1, 21))
accuracies = []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train, y_train)
    accuracies.append(accuracy_score(y_test, m.predict(X_test)))

best_acc = max(accuracies)
best_k = max(k for k, a in zip(k_values, accuracies) if a == best_acc)
print(f'Best k = {best_k} with accuracy = {best_acc:.4f}')

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, accuracies, marker='o', label='Accuracy')
plt.scatter([best_k], [best_acc], color='red', s=90, zorder=5, label=f'Best k = {best_k}')
plt.axvline(best_k, color='red', linestyle='--', alpha=0.5)
plt.xticks(k_values)
plt.xlabel('Number of Neighbours (k)'); plt.ylabel('Test Accuracy')
plt.title('Accuracy vs. K Value'); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()

### Retrain with the best k and compare

In [ ]:
best_model = KNeighborsClassifier(n_neighbors=best_k)
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

default_acc = accuracy_score(y_test, y_pred)
tuned_acc = accuracy_score(y_test, best_pred)
print(f'Default (k=5)  accuracy: {default_acc:.4f}')
print(f'Best (k={best_k}) accuracy: {tuned_acc:.4f}')
print(f'Improvement: {tuned_acc - default_acc:+.4f}')

## Conclusion

KNN classifies the Iris species with high accuracy. Feature scaling ensures each measurement contributes fairly to the distance calculation, the train/test split gives an honest estimate of performance, and tuning `k` squeezes out a little extra accuracy. See the modular version of this pipeline in the `src/` folder and run it end to end with `python main.py`.